In [5]:
# ==============================================================================
# Task 1 — Data Loading & Exploration
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
df = pd.read_csv("HR_Attrition.csv")

# Display the first 10 rows
print("--- First 10 Rows ---")
print(df.head(10))

# Dimensions of the dataset
rows, cols = df.shape
print(f"\nDataset contains {rows} rows and {cols} columns.")

# Target distribution & Attrition Rate
attrition_counts = df['Attrition'].value_counts()
print("\nEmployee Retention Summary:")
print(attrition_counts)

attrition_rate = (attrition_counts['Yes'] / rows) * 100
print(f"Overall Attrition Rate: {attrition_rate:.2f}%")

# Identify numeric vs categorical features
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"\nNumerical Columns count: {len(numeric_cols)}")
print(f"Categorical Columns count: {len(categorical_cols)}")

--- First 10 Rows ---
   Age Attrition     BusinessTravel  DailyRate              Department  \
0   41       Yes      Travel_Rarely       1102                   Sales   
1   49        No  Travel_Frequently        279  Research & Development   
2   37       Yes      Travel_Rarely       1373  Research & Development   
3   33        No  Travel_Frequently       1392  Research & Development   
4   27        No      Travel_Rarely        591  Research & Development   
5   32        No  Travel_Frequently       1005  Research & Development   
6   59        No      Travel_Rarely       1324  Research & Development   
7   30        No      Travel_Rarely       1358  Research & Development   
8   38        No  Travel_Frequently        216  Research & Development   
9   36        No      Travel_Rarely       1299  Research & Development   

   DistanceFromHome  Education EducationField  EmployeeCount  EmployeeNumber  \
0                 1          2  Life Sciences              1               1   
1  

/tmp/ipykernel_10921/4269944675.py:30: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object']).columns.tolist()


In [6]:
# ==============================================================================
# Task 2 — Data Cleaning & Preprocessing
# ==============================================================================
from sklearn.preprocessing import StandardScaler

# Check for missing values
print("\nMissing values per column:\n", df.isnull().sum().sum())

# Drop irrelevant / zero-variance constant columns
cols_to_drop = ['EmployeeNumber', 'Over18', 'StandardHours', 'EmployeeCount']
df_clean = df.drop(columns=cols_to_drop, errors='ignore')

# Map binary target column to numerical digits
df_clean['Attrition'] = df_clean['Attrition'].map({'Yes': 1, 'No': 0})

# Segregate data features from the target label
X = df_clean.drop(columns=['Attrition'])
y = df_clean['Attrition']

# Encode multi-class categorical indicators natively using One-Hot encoding
X_encoded = pd.get_dummies(X, drop_first=True)

# Standardize feature scale for numerical elements
scaler = StandardScaler()
# Extract modern, fresh columns list to scale after string features are encoded
continuous_features = [col for col in numeric_cols if col in X_encoded.columns]
X_encoded[continuous_features] = scaler.fit_transform(X_encoded[continuous_features])


Missing values per column:
 0


In [7]:
# ==============================================================================
# Task 3 & 6 — EDA & Visualization Pipeline
# ==============================================================================
sns.set_theme(style="whitegrid")

# Chart 1: Attrition rates by Department and Job Role
plt.figure(figsize=(14, 6))
sns.barplot(data=df, x='JobRole', y=df['Attrition'].map({'Yes':1, 'No':0}), hue='Department', ci=None, palette='muted')
plt.xticks(rotation=45, ha='right')
plt.title('Attrition Rate by Job Role and Corporate Department')
plt.ylabel('Attrition Proportion')
plt.tight_layout()
plt.savefig('charts/attrition_by_role.png')
plt.close()

# Chart 2: Monthly Income comparisons
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='Attrition', y='MonthlyIncome', palette='Set2')
plt.title('Monthly Income Distribution: Departures vs. Retained Staff')
plt.xticks([0, 1], ['Stayed (0)', 'Left (1)'])
plt.savefig('charts/income_vs_attrition.png')
plt.close()

/tmp/ipykernel_10921/1015421757.py:8: FutureWarning: 

The `ci` parameter is deprecated. Use `errorbar=None` for the same effect.

  sns.barplot(data=df, x='JobRole', y=df['Attrition'].map({'Yes':1, 'No':0}), hue='Department', ci=None, palette='muted')
/tmp/ipykernel_10921/1015421757.py:18: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x='Attrition', y='MonthlyIncome', palette='Set2')


In [8]:
# ==============================================================================
# Task 4 & 5 — Model Building, Training & Evaluation
# ==============================================================================
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, roc_curve

# Split train/test sets
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.20, random_state=42, stratify=y)

# Instantiate models
models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(class_weight='balanced', random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42) # Custom handling via thresholding or weights if needed
}

results_summary = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    
    results_summary[name] = {
        'ROC-AUC': roc_auc_score(y_test, probs),
        'Matrix': confusion_matrix(y_test, preds),
        'Report': classification_report(y_test, preds, output_dict=True)
    }

# Chart 3: Confusion Matrix Heatmap for Best Performing Baseline (Example: Logistic Regression)
best_cm = results_summary['Logistic Regression']['Matrix']
plt.figure(figsize=(5, 4))
sns.heatmap(best_cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Stayed', 'Left'], yticklabels=['Stayed', 'Left'])
plt.title('Confusion Matrix Heatmap')
plt.ylabel('True State')
plt.xlabel('Predicted State')
plt.savefig('charts/confusion_matrix_best.png')
plt.close()

# Chart 4: Top 10 Feature Importances (Using Logistic Regression weights for explanation)
log_model = models['Logistic Regression']
importances = np.abs(log_model.coef_[0])
feat_importances = pd.Series(importances, index=X_encoded.columns).sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=feat_importances.values, y=feat_importances.index, palette='viridis')
plt.title('Top 10 Drivers of Attrition (Model Feature Weights)')
plt.xlabel('Relative Absolute Impact Score')
plt.savefig('charts/feature_importance.png')
plt.close()

/tmp/ipykernel_10921/1361559881.py:48: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=feat_importances.values, y=feat_importances.index, palette='viridis')
